# سبق 08 - کثیر ایجنٹ ڈیزائن پیٹرن


## سیٹ اپ


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ملٹی ایجنٹ سسٹمز کیوں؟

حقیقی دنیا کے کام جیسے کہ سفر کی منصوبہ بندی میں مختلف قسم کی مہارتیں شامل ہوتی ہیں — لاجسٹکس، مقامی معلومات، بجٹنگ، اور بہت کچھ۔ ایک واحد ایجنٹ جو سب کچھ سنبھالنے کی کوشش کرتا ہے، جلد ہی ناقابلِ قابو ہو جاتا ہے۔

ملٹی ایجنٹ سسٹمز اس مسئلے کو **تخصص** کے ذریعے حل کرتے ہیں: ہر ایجنٹ ایک مخصوص مہارت کے علاقے پر توجہ مرکوز کرتا ہے، جو کہ ایک عام ایجنٹ سے بہتر معیار کے نتائج فراہم کرتا ہے۔ یہ **پیمائش پذیری** کو بھی بہتر بناتے ہیں — آپ نئے ایجنٹس (مثلاً، ایک فلائٹ اسپیشلسٹ، ایک ریسٹورنٹ نقاد) کو موجودہ ورک فلو کو دوبارہ لکھے بغیر شامل کر سکتے ہیں۔ ایجنٹس ایک منظم پائپ لائن کے ذریعے آپس میں جڑے ہوتے ہیں، جس میں ایک سے دوسرے تک سیاق و سباق منتقل کیا جاتا ہے۔


## خصوصی ایجنٹس بنانا


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## سلسلہ وار ورک فلو تیار کرنا

`WorkflowBuilder` آپ کو ایجنٹس کو ایک ہدایت یافتہ گراف میں مربوط کرنے دیتا ہے۔ یہاں ہم ایک سادہ دو مرحلوں پر مشتمل پائپ لائن بناتے ہیں: **TravelPlanner** سفر کا منصوبہ بناتا ہے، پھر **TravelConcierge** اسے دیکھتا اور بہتر بناتا ہے۔


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ورک فلو میں مزید ایجنٹس شامل کرنا

ملٹی ایجنٹ پیٹرن کا ایک بڑا فائدہ یہ ہے کہ اسے بڑھانا کتنا آسان ہے۔ نیچے ہم ایک **BudgetReviewer** ایجنٹ شامل کرتے ہیں جو منصوبے کو مسافر کے بجٹ کے خلاف چیک کرتا ہے، ایسی اشیاء کو نشان زد کرتا ہے جو خرچ کو حد سے اوپر لے جا سکتی ہیں، اور پیسہ بچانے کے متبادلات تجویز کرتا ہے۔ ورک فلو اب تین ایجنٹس کو ترتیب سے چلاتا ہے:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## خلاصہ

اس سبق میں آپ نے سیکھا کہ کیسے:

1. **ماہرانہ ایجنٹس تخلیق کریں** — ہر ایک کا مخصوص کردار ہو (منصوبہ بندی، کنسیرج، بجٹ جائزہ)۔
2. **ایجنٹس کو ترتیبی ورک فلو میں جوڑیں** `WorkflowBuilder` اور `add_edge` استعمال کرتے ہوئے۔
3. **کئی ایجنٹس والی پائپ لائن سے آؤٹ پٹ کو اسٹریمنگ کریں**، معلوم کریں کہ کون سا ایجنٹ بول رہا ہے۔
4. **ورک فلو کو بڑھائیں** نئے ایجنٹس کو چین میں شامل کرکے بغیر موجودہ ایجنٹس میں ترمیم کیے۔

کثیر ایجنٹس کا ڈیزائن پیٹرن ہر ایجنٹ کو سادہ رکھتا ہے جبکہ ایک سے بڑھ کر ایجنٹ کے ذریعے زیادہ جامع، بہتر طور پر جائزہ شدہ نتائج پیدا کرتا ہے جنہیں کوئی واحد ایجنٹ تنہا حاصل نہیں کر سکتا۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
